# Exploratory Data Analysis

## Introduction

*Centers for Medicare & Medicaid Services (CMS)* is a U.S. federal government agency responsible for administering major public healthcare programs, including Medicare.

**Medicare** is a U.S. federal health insurance program primarily serving people aged 65 and older. Eligibility can also extend to certain people under 65 with qualifying disabilities, End-Stage Renal Disease (ESRD), or Amyotrophic Lateral Sclerosis (ALS).

Medicare is divided into four parts. This project focuses on **Medicare Part B fee-for-service (FFS) data**.

**Medicare Part B** covers a wide range of medically necessary services provided by physicians and other healthcare professionals, including physician services, outpatient services, preventive services, and diagnostic services. Beneficiaries generally pay a monthly Part B premium and an annual deductible. After the deductible, beneficiaries generally pay 20% of the Medicare-approved amount for covered services when the applicable Medicare payment rules are met.

This project uses the **CMS Medicare Physician & Other Practitioners by Provider and Service** dataset to analyze provider-, service-, specialty-, and regional-level patterns in Medicare healthcare services and payments.


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import os
warnings.filterwarnings('ignore')

SAMPLE_SIZE = 200_000

# Navigate up from wherever Jupyter is to find the project root
PROJECT_ROOT = Path(os.getcwd())
while not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FILE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022.csv'

df_full = pd.read_csv(
    FILE_PATH,
    usecols=[
        'Rndrng_NPI', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_State_Abrvtn',
        'Rndrng_Prvdr_Ent_Cd', 'HCPCS_Cd', 'Place_Of_Srvc',
        'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs',
        'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt'
    ],
    low_memory=False
)

# Now take a proper random sample
df = df_full.sample(n=SAMPLE_SIZE, random_state=42)

print(f"Full dataset rows: {len(df_full):,}")
print(f"Sample rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Full dataset rows: 9,755,427
Sample rows: 200,000
Columns: 13


In [2]:
df.head(3)

,Rndrng_NPI,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
3351656,1346268067,I,CA,Infectious Disease,99291,F,19,20.0,20,970.000000,233.674500,188.455500,173.877500
4171590,1427314038,I,NV,Diagnostic Radiology,72125,F,13,13.0,13,568.000000,47.860769,29.512308,28.015385
4604675,1477188373,I,TX,Nurse Practitioner,90662,O,17,17.0,17,67.764706,66.465882,66.465882,66.409412


## Section 2: Missingness & Null Value Assessment
Profiling null values across the 13 analytical columns. All columns show 0 missing values, confirming that CMS provider/service aggregate records enforce complete data reporting for required fee-for-service fields.

In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 200000 entries, 3351656 to 4048291
Data columns (total 13 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Rndrng_NPI                 200000 non-null  int64  
 1   Rndrng_Prvdr_Ent_Cd        200000 non-null  str    
 2   Rndrng_Prvdr_State_Abrvtn  200000 non-null  str    
 3   Rndrng_Prvdr_Type          200000 non-null  str    
 4   HCPCS_Cd                   200000 non-null  str    
 5   Place_Of_Srvc              200000 non-null  str    
 6   Tot_Benes                  200000 non-null  int64  
 7   Tot_Srvcs                  200000 non-null  float64
 8   Tot_Bene_Day_Srvcs         200000 non-null  int64  
 9   Avg_Sbmtd_Chrg             200000 non-null  float64
 10  Avg_Mdcr_Alowd_Amt         200000 non-null  float64
 11  Avg_Mdcr_Pymt_Amt          200000 non-null  float64
 12  Avg_Mdcr_Stdzd_Amt         200000 non-null  float64
dtypes: float64(5), int64(3), str(5)
memory

## Financial column ranges and Negative value check

In [9]:
# Cell 5 — Numeric summary statistics (Financial + Volume)
numeric_cols = [
    'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 
    'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt',
    'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs'
]
print(df_full[numeric_cols].describe().round(2))

       Avg_Sbmtd_Chrg  Avg_Mdcr_Alowd_Amt  Avg_Mdcr_Pymt_Amt  \
count      9755427.00          9755427.00         9755427.00   
mean           398.99              104.98              81.95   
std           1358.12              312.13             249.16   
min              0.00                0.00               0.00   
25%             69.00               27.32              21.31   
50%            166.67               68.54              52.28   
75%            343.00              120.67              89.22   
max          99999.99            59035.73           47043.99   

       Avg_Mdcr_Stdzd_Amt   Tot_Benes   Tot_Srvcs  Tot_Bene_Day_Srvcs  
count          9755427.00  9755427.00  9755427.00          9755427.00  
mean                80.93       84.27      258.22              134.31  
std                248.43     1301.98     6269.63             2116.34  
min                  0.00       11.00        2.50               11.00  
25%                 19.52       17.00       20.00              

In [8]:
# Cell 6 — Boundary checks (Negative & Zero values)
for col in numeric_cols:
    neg_count = (df_full[col] < 0).sum()
    zero_count = (df_full[col] == 0).sum()
    print(f"{col:20s}: {neg_count} negatives | {zero_count} zeroes")

Avg_Sbmtd_Chrg      : 0 negatives | 0 zeroes
Avg_Mdcr_Alowd_Amt  : 0 negatives | 15 zeroes
Avg_Mdcr_Pymt_Amt   : 0 negatives | 109 zeroes
Avg_Mdcr_Stdzd_Amt  : 0 negatives | 25 zeroes
Tot_Benes           : 0 negatives | 0 zeroes
Tot_Srvcs           : 0 negatives | 0 zeroes
Tot_Bene_Day_Srvcs  : 0 negatives | 0 zeroes


### Section 4: Categorical Domain Profiling

In [7]:
cat_cols = ['Rndrng_Prvdr_Type', 'Place_Of_Srvc', 'Rndrng_Prvdr_Ent_Cd']
for col in cat_cols:
    print(f"{col}: {df_full[col].nunique()} unique values")
    print(df_full[col].value_counts().head(5))
    print()

Rndrng_Prvdr_Type: 103 unique values
Rndrng_Prvdr_Type
Diagnostic Radiology    1137764
Internal Medicine        868292
Nurse Practitioner       851917
Family Practice          813473
Physician Assistant      518182
Name: count, dtype: int64

Place_Of_Srvc: 2 unique values
Place_Of_Srvc
O    6233245
F    3522182
Name: count, dtype: int64

Rndrng_Prvdr_Ent_Cd: 2 unique values
Rndrng_Prvdr_Ent_Cd
I    9210142
O     545285
Name: count, dtype: int64

